In [1]:
# ============================================================
# 0. 基本配置
# ============================================================
import json
import csv
from pathlib import Path

import pandas as pd

# 这里改成你自己的 json 目录
BASE_DIR = Path(r"F:\iot\datasets\Okui22\KDDI-IoT-2019-main\KDDI-IoT-2019-main\ipfix\json")

# 所有 json 文件
json_files = sorted(BASE_DIR.glob("*.json"))
print("Found json files:", len(json_files))

# 标准 30 分钟窗口
WINDOW = 1800

SERVICE_BUCKETS = ["DNS", "HTTP", "NTP", "SMB", "HTTPS", "SSDP", "TCP_OTHER", "UDP_OTHER"]
ELEMENTS = ["octetTotalCount", "reverseOctetTotalCount",
            "packetTotalCount", "reversePacketTotalCount"]
STATS = ["max", "min", "mean", "median"]


def map_service(port, proto):
    """按端口 + 协议映射到 Okui22 的服务桶"""
    if port in (53, 5353):
        return "DNS"
    elif port in (80, 8008, 8080, 8888):
        return "HTTP"
    elif port == 123:
        return "NTP"
    elif port in (137, 138, 445):
        return "SMB"
    elif port in (443, 1443, 8443, 55443):
        return "HTTPS"
    elif port == 1900:
        return "SSDP"
    else:
        return "UDP_OTHER" if proto == 17 else "TCP_OTHER"


# ============================================================
# 1. 大文件：流式提取精简 flows CSV
# ============================================================
def extract_minimal_flows(json_path: Path, out_csv: Path):
    """
    针对超大 NDJSON：
    一行一行读，展开 flows，只保留后续需要的少数几列，
    写成一个精简版 flows CSV。
    """
    print(f"[*] extract_minimal_flows: {json_path.name} -> {out_csv.name}")

    with json_path.open("r", encoding="utf-8") as fin, \
            out_csv.open("w", newline="", encoding="utf-8") as fout:

        writer = csv.writer(fout)
        writer.writerow([
            "flowEndMilliseconds",   # 字符串时间戳
            "dst_port",              # 目的/源端口
            "proto",                 # 协议号
            "octetTotalCount",
            "reverseOctetTotalCount",
            "packetTotalCount",
            "reversePacketTotalCount",
        ])

        n = 0
        for line in fin:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            flow = obj.get("flows", {})
            if not flow:
                continue

            end_ts = flow.get("flowEndMilliseconds")

            # 端口：优先目的端口，没有就用源端口
            dst_port = flow.get("destinationTransportPort",
                                flow.get("sourceTransportPort"))

            proto = flow.get("protocolIdentifier")

            octet = flow.get("octetTotalCount", 0.0)
            pkt = flow.get("packetTotalCount", 0.0)

            # 反向 bytes：KDDI 里多叫 reverseDataByteCount
            rev_octet = flow.get("reverseDataByteCount",
                                 flow.get("reverseOctetTotalCount", 0.0))

            # 反向 packets：没有统一字段就先用 0
            rev_pkt = flow.get("reversePacketTotalCount", 0.0)

            writer.writerow([
                end_ts, dst_port, proto,
                octet, rev_octet, pkt, rev_pkt
            ])

            n += 1
            if n % 500000 == 0:
                print(f"  ... wrote {n} rows")

    print(f"[+] Done extract_minimal_flows, total rows: {n}")


# ============================================================
# 2. 大文件：从精简 flows CSV 生成 136D 特征
# ============================================================
def features_from_flows_csv(device_id: str, flows_csv: Path, out_csv: Path):
    print(f"[*] features_from_flows_csv: {flows_csv.name} -> {out_csv.name}")

    df = pd.read_csv(flows_csv)

    # 基本清洗
    df["dst_port"] = df["dst_port"].fillna(0).astype(int)
    df["proto"] = df["proto"].fillna(0).astype(int)

    # 字符串时间戳 -> datetime
    df["random_ts"] = pd.to_datetime(
        df["flowEndMilliseconds"],
        errors="coerce",
    )

    # 统一字段名
    df["octetTotalCount"] = df["octetTotalCount"].fillna(0.0)
    df["reverseOctetTotalCount"] = df["reverseOctetTotalCount"].fillna(0.0)
    df["packetTotalCount"] = df["packetTotalCount"].fillna(0.0)
    df["reversePacketTotalCount"] = df["reversePacketTotalCount"].fillna(0.0)

    # 服务桶
    df["service"] = [
        map_service(p, pr)
        for p, pr in zip(df["dst_port"], df["proto"])
    ]

    # 时间窗
    t0 = df["random_ts"].min()
    df["window_id"] = ((df["random_ts"] - t0).dt.total_seconds() // WINDOW).astype(int)

    # 统计
    agg = {"count": ("octetTotalCount", "size")}
    for e in ELEMENTS:
        for s in STATS:
            agg[f"{e}_{s}"] = (e, s)

    grp = (
        df.groupby(["window_id", "service"])
        .agg(**agg)
        .reset_index()
    )

    # pivot 到 136D
    vals = grp.columns.difference(["window_id", "service"])
    wide = grp.pivot(index="window_id", columns="service", values=vals)
    wide.columns = [f"{srv}__{feat}" for feat, srv in wide.columns]
    wide.reset_index(inplace=True)

    # 补齐所有维度
    expected = []
    for srv in SERVICE_BUCKETS:
        expected.append(f"{srv}__count")
        for e in ELEMENTS:
            for s in STATS:
                expected.append(f"{srv}__{e}_{s}")

    for c in expected:
        if c not in wide.columns:
            wide[c] = 0.0

    wide["device_id"] = device_id
    wide = wide[["device_id", "window_id"] + expected]
    wide = wide.fillna(0.0)

    wide.to_csv(out_csv, index=False)
    print(f"[+] Saved {out_csv.name}  windows={len(wide)}")


# ============================================================
# 3. 小文件：直接 NDJSON -> 136D
# ============================================================
def process_one_json(json_file: Path):
    """
    小 JSON 用 pandas 一次性读入，展开 flows，再做 136D 特征。
    """
    device_id = json_file.stem
    out_csv = json_file.parent / f"{device_id}_136d.csv"

    print(f"[*] Processing (small) {device_id}")

    outer = pd.read_json(json_file, lines=True)
    outer = outer.dropna(subset=["flows"])
    df = pd.json_normalize(outer["flows"])

    # 端口：优先目的端口，没有就用源端口
    if "destinationTransportPort" in df.columns:
        df["dst_port"] = df["destinationTransportPort"].astype(int)
    elif "sourceTransportPort" in df.columns:
        df["dst_port"] = df["sourceTransportPort"].astype(int)
    else:
        df["dst_port"] = 0

    df["proto"] = df["protocolIdentifier"].astype(int)

    # 时间（字符串）
    df["random_ts"] = pd.to_datetime(
        df["flowEndMilliseconds"],
        errors="coerce",
    )

    # 4 个 IPFIX 元素
    df["octetTotalCount"] = df["octetTotalCount"]
    df["packetTotalCount"] = df["packetTotalCount"]

    if "reverseDataByteCount" in df.columns:
        df["reverseOctetTotalCount"] = df["reverseDataByteCount"]
    elif "reverseOctetTotalCount" in df.columns:
        df["reverseOctetTotalCount"] = df["reverseOctetTotalCount"]
    else:
        df["reverseOctetTotalCount"] = 0.0

    if "reversePacketTotalCount" in df.columns:
        df["reversePacketTotalCount"] = df["reversePacketTotalCount"]
    else:
        df["reversePacketTotalCount"] = 0.0

    # 服务桶
    df["service"] = [
        map_service(p, pr)
        for p, pr in zip(df["dst_port"], df["proto"])
    ]

    # 时间窗
    t0 = df["random_ts"].min()
    df["window_id"] = ((df["random_ts"] - t0).dt.total_seconds() // WINDOW).astype(int)

    # 统计
    agg = {"count": ("octetTotalCount", "size")}
    for e in ELEMENTS:
        for s in STATS:
            agg[f"{e}_{s}"] = (e, s)

    grp = (
        df.groupby(["window_id", "service"])
        .agg(**agg)
        .reset_index()
    )

    vals = grp.columns.difference(["window_id", "service"])
    wide = grp.pivot(index="window_id", columns="service", values=vals)
    wide.columns = [f"{srv}__{feat}" for feat, srv in wide.columns]
    wide.reset_index(inplace=True)

    # 补齐维度
    expected = []
    for srv in SERVICE_BUCKETS:
        expected.append(f"{srv}__count")
        for e in ELEMENTS:
            for s in STATS:
                expected.append(f"{srv}__{e}_{s}")

    for c in expected:
        if c not in wide.columns:
            wide[c] = 0.0

    wide["device_id"] = device_id
    wide = wide[["device_id", "window_id"] + expected]
    wide = wide.fillna(0.0)

    wide.to_csv(out_csv, index=False)
    print(f"[+] Saved {out_csv.name}  rows={len(wide)}")


# ============================================================
# 4. 批处理：自动区分大/小文件，生成所有 *_136d.csv
# ============================================================

# 你认为“太大”的 json 文件（按文件名）
LARGE_FILES = {
    "google_home_gen1.json",
    "i-o_data_qwatch.json",
    "sony_bravia.json",
    "jvc_kenwood_cu-hb1.json",
    "panasonic_doorphone.json",
    "sony_smart_speaker.json",
}

for jf in json_files:
    device_id = jf.stem
    out_136d = BASE_DIR / f"{device_id}_136d.csv"

    # 已经算过的直接跳过
    if out_136d.exists():
        print(f"[skip] {device_id} already has 136d.csv")
        continue

    # 大文件：流式流程
    if jf.name in LARGE_FILES:
        print(f"\n[big] processing {device_id}")
        tmp_flows_csv = BASE_DIR / f"{device_id}_flows_min.csv"

        if not tmp_flows_csv.exists():
            extract_minimal_flows(jf, tmp_flows_csv)

        features_from_flows_csv(device_id, tmp_flows_csv, out_136d)

    # 小文件：直接 NDJSON -> 136D
    else:
        print(f"\n[small] processing {device_id}")
        process_one_json(jf)


# ============================================================
# 5. 合并所有 136D 特征成一个总 CSV
# ============================================================

all_136d = sorted(BASE_DIR.glob("*_136d.csv"))
print("136d files:", len(all_136d))

dfs = [pd.read_csv(f) for f in all_136d]
merged = pd.concat(dfs, ignore_index=True)

out_merged = BASE_DIR / "kddi_okui22_all_devices_136d.csv"
merged.to_csv(out_merged, index=False)

print("Merged file:", out_merged)
print("Merged shape:", merged.shape)


Found json files: 25
[skip] amazon_echo_gen2 already has 136d.csv
[skip] au_network_camera already has 136d.csv
[skip] au_wireless_adapter already has 136d.csv
[skip] bitfinder_awair_breathe_easy already has 136d.csv
[skip] candy_house_sesami_wi-fi_access_point already has 136d.csv
[skip] google_home_gen1 already has 136d.csv
[skip] i-o_data_qwatch already has 136d.csv
[skip] irobot_roomba already has 136d.csv
[skip] jvc_kenwood_cu-hb1 already has 136d.csv
[skip] jvc_kenwood_hdtv_ip_camera already has 136d.csv
[skip] line_clova_wave already has 136d.csv
[skip] link_japan_eremote already has 136d.csv
[skip] mouse_computer_room_hub already has 136d.csv
[skip] nature_remo already has 136d.csv
[skip] panasonic_doorphone already has 136d.csv
[skip] philips_hue_bridge already has 136d.csv
[skip] planex_camera_one_shot! already has 136d.csv
[skip] planex_smacam_outdoor already has 136d.csv
[skip] planex_smacam_pantilt already has 136d.csv
[skip] powerelectric_wi-fi_plug already has 136d.csv
[

In [3]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path(r"F:\iot\datasets\Okui22\KDDI-IoT-2019-main\KDDI-IoT-2019-main\ipfix\json\kddi_okui22_all_devices_136d.csv"))

print("Columns:", len(df.columns))
print(df.columns[:20])  # 看前 20 列

services   = set(col.split("__")[0] for col in df.columns if "__" in col)
elements   = set(c.split("__")[1].rsplit("_", 2)[0] for c in df.columns if "__" in c and "count" not in c)
print("services:", services)
print("elements:", elements)



Columns: 138
Index(['device_id', 'window_id', 'DNS__count', 'DNS__octetTotalCount_max',
       'DNS__octetTotalCount_min', 'DNS__octetTotalCount_mean',
       'DNS__octetTotalCount_median', 'DNS__reverseOctetTotalCount_max',
       'DNS__reverseOctetTotalCount_min', 'DNS__reverseOctetTotalCount_mean',
       'DNS__reverseOctetTotalCount_median', 'DNS__packetTotalCount_max',
       'DNS__packetTotalCount_min', 'DNS__packetTotalCount_mean',
       'DNS__packetTotalCount_median', 'DNS__reversePacketTotalCount_max',
       'DNS__reversePacketTotalCount_min', 'DNS__reversePacketTotalCount_mean',
       'DNS__reversePacketTotalCount_median', 'HTTP__count'],
      dtype='object')
services: {'DNS', 'HTTP', 'SMB', 'TCP_OTHER', 'SSDP', 'UDP_OTHER', 'NTP', 'HTTPS'}
elements: {'packetTotalCount', 'reversePacketTotalCount', 'octetTotalCount', 'reverseOctetTotalCount'}
